
 ## 1.) Se importan las librerías necesarias para el entrenamiento del modelo

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping

## 2) Se realiza el cargue del archivo de datos, realizando la lectura de los datos con pandas

In [2]:
url = "https://github.com/adiacla/bigdata/raw/master/riesgo.xlsx"
df = pd.read_excel(url)

## 3) Realizamos la inspección de los datos, para conocer la cantidad de filas y columnas, los tipos de datos etc.

In [3]:
print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())

  Customer_ID            Name     Age          SSN     Occupation  \
0  CUS_0x1000  Alistair Barrf  17.375  913-74-1218         Lawyer   
1  CUS_0x1009          Arunah  25.750  063-67-6938       Mechanic   
2  CUS_0x100b        Shirboni  18.500  238-62-0395  Media_Manager   
3  CUS_0x1011       Schneyerh  43.875  793-05-8223         Doctor   
4  CUS_0x1013        Cameront  43.750  930-49-9615       Mechanic   

   Annual_Income  Monthly_Inhand_Salary  Num_Bank_Accounts  Num_Credit_Card  \
0       30625.94            2706.161667                6.0              5.0   
1       52312.68            4250.390000                6.0              5.0   
2      113781.39            9549.782500                1.0              4.0   
3       58918.47            5208.872500                3.0              3.0   
4       98620.98            7962.415000                3.0              3.0   

   Interest_Rate  ...  Credit_Mix Outstanding_Debt  Credit_Utilization_Ratio  \
0             27  ...         

In [4]:
print(df.columns.tolist())

['Customer_ID', 'Name', 'Age', 'SSN', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Type_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Payment_of_Min_Amount', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance', 'Credit_Score']


## 4) Se realiza la eliminación de columnas inncesesarias.
Teniendo en cuenta el objetivo que se busca con el entrenamiento del modelo, se validan columnas las cuales no deben entrar al modelo porque son identificadores, datos sensibles o no predictivos, como es el caso de:
- Customer_ID
- SSN
- Name

Estas columnas se eliminan puesto que no aportan valor predictivo en el contexto actual.

In [5]:
columnas_eliminar = ["Customer_ID", "SSN", "Name"]
df = df.drop(columns=columnas_eliminar)

## 5) Se realiza la separación de los datos en variables predictoras (X) y variable objetivo (y)

In [6]:
X = df.drop(columns=["Credit_Score"])
y = df["Credit_Score"]

# Verificación de la distribución
print(y.value_counts())

Credit_Score
1    6111
0    4162
2    2227
Name: count, dtype: int64


## 6) Identificación columnas
Se realiza la identificación de las columnas numéricas y categóricas para aplicar el preprocesamiento adecuado a cada tipo de dato.

In [7]:
columnas_numericas = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
columnas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()

print("Numéricas:", columnas_numericas)
print("Categóricas:", columnas_categoricas)

Numéricas: ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance']
Categóricas: ['Occupation', 'Type_of_Loan', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']


C:\Users\daelr\AppData\Local\Temp\ipykernel_66788\3259653239.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()


## 7) División de datos
Se realiza la división de los datos en conjuntos de entrenamiento y prueba utilizando la función train_test_split de sklearn, con la siguiente proporción:
- 60% para entrenamiento
- 20% para validación
- 20% para prueba


In [8]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    random_state=42,
    stratify=y_train_full
)